In [1]:
import torch
import torch.nn as nn 
import numpy as np 
import pandas as pd
import os ,  random , joblib 
from collections  import   deque
import copy
import torch.optim as optim
from pathlib import Path

In [2]:

device = "cuda"


In [3]:
NUM_CORES = 16 
PREDICTED_UTIL_FEATURES = 16 
CURRENT_UTIL_FEATURES = 16 
QUEUE_FEATURES = 16
TASK_FEATURES = 4 

STATE_DIM = ( PREDICTED_UTIL_FEATURES  + CURRENT_UTIL_FEATURES +  QUEUE_FEATURES +  TASK_FEATURES)
ACTION_DIM = NUM_CORES

In [4]:
class DQNetwork(nn.Module) : 
    def __init__(self , state_dim = STATE_DIM , action_dim = ACTION_DIM ) : 
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear( state_dim , 128) , 
            nn.ReLU() , 
            nn.Linear(128 , 128 ) , 
            nn.ReLU() , 
            nn.Linear(128 , action_dim)
        )
    def forward(self , state) : 
        return self.network(state)

In [5]:
dqn = DQNetwork().to(device)

In [6]:
def select_action(dqn, state, epsilon=0.0, valid_actions=None):
    if not 0.0 <= epsilon <= 1.0:
        raise ValueError("epsilon must be between 0 and 1")
    if valid_actions is None:
        valid_actions = list(range(ACTION_DIM))
    valid_actions = np.asarray(list(valid_actions), dtype=np.int64)
    if valid_actions.size == 0:
        raise ValueError("No eligible CPU cores available.")
    if np.any((valid_actions < 0) | (valid_actions >= ACTION_DIM)):
        raise ValueError(f"Eligible core index must be between 0 and {ACTION_DIM - 1}.")
    if random.random() < epsilon:
        return int(random.choice(valid_actions.tolist()))
    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).reshape(1, STATE_DIM)
    was_training = dqn.training
    dqn.eval()
    with torch.no_grad():
        q_values = dqn(state_tensor).squeeze(0)
        mask = torch.zeros(ACTION_DIM, dtype=torch.bool, device=device)
        mask[torch.as_tensor(valid_actions, dtype=torch.long, device=device)] = True
        q_values = q_values.masked_fill(~mask, -torch.inf)
        action = int(q_values.argmax().item())
    dqn.train(was_training)
    return action


In [7]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.memory = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done, next_valid_actions=None):
        if next_valid_actions is None:
            next_valid_actions = np.ones(ACTION_DIM, dtype=np.bool_)
        else:
            mask = np.zeros(ACTION_DIM, dtype=np.bool_)
            indices = np.asarray(list(next_valid_actions), dtype=np.int64)
            if indices.size == 0 or np.any((indices < 0) | (indices >= ACTION_DIM)):
                raise ValueError("next_valid_actions must contain valid core indices")
            mask[indices] = True
            next_valid_actions = mask
        state = np.asarray(state, dtype=np.float32).reshape(STATE_DIM)
        next_state = np.asarray(next_state, dtype=np.float32).reshape(STATE_DIM)
        if not np.isfinite(state).all() or not np.isfinite(next_state).all():
            raise ValueError("Replay states must contain only finite values")
        self.memory.append((state, int(action), float(reward), next_state, bool(done), next_valid_actions))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones, next_masks = zip(*batch)
        return (np.stack(states), np.asarray(actions, dtype=np.int64),
                np.asarray(rewards, dtype=np.float32), np.stack(next_states),
                np.asarray(dones, dtype=np.float32), np.stack(next_masks))

    def __len__(self):
        return len(self.memory)


In [8]:
replay_buffer = ReplayBuffer(capacity= 10000)

In [9]:
policy_net = dqn.to(device)

target_net = copy.deepcopy(policy_net).to(device)
target_net.eval()
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
loss_function = nn.SmoothL1Loss()
gamma = 0.99


In [10]:
def train_step(policy_net, target_net, optimizer, buffer, batch_size=32, gamma=0.99):
    if len(buffer) < batch_size:
        return None
    states, actions, rewards, next_states, dones, next_masks = buffer.sample(batch_size)
    
    states = torch.as_tensor(states, dtype=torch.float32, device=device)
    actions = torch.as_tensor(actions, dtype=torch.long, device=device)
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)
    next_states = torch.as_tensor(next_states, dtype=torch.float32, device=device)
    dones = torch.as_tensor(dones, dtype=torch.float32, device=device)
    next_masks = torch.as_tensor(next_masks, dtype=torch.bool, device=device)
    
    current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        
        next_policy_q = policy_net(next_states).masked_fill(~next_masks, -torch.inf)
        no_valid_next = ~next_masks.any(dim=1)
        next_policy_q[no_valid_next] = 0.0
        
        next_actions = next_policy_q.argmax(dim=1, keepdim=True)
        next_q = target_net(next_states).gather(1, next_actions).squeeze(1)
        next_q = next_q.masked_fill(no_valid_next, 0.0)
        
        target_q = rewards + gamma * (1.0 - dones) * next_q
        
    loss = loss_function(current_q, target_q)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
    optimizer.step()
    
    return float(loss.item())


In [11]:
project_root = Path.cwd()
if project_root.name == "DQN_training":
    project_root = project_root.parent
lstm_dir = project_root / "LSTM_training"
model_path = lstm_dir / "cpu_lstm_model.pth"
feature_scaler_path = lstm_dir / "cpu_feature_scaler.pkl"
target_scaler_path = lstm_dir / "cpu_target_scaler.pkl"
feature_scaler = joblib.load(feature_scaler_path)
target_scaler = joblib.load(target_scaler_path)


In [12]:
class cpu_utilizationlstm(nn.Module):
    def __init__(
        self,
        input_size=35,
        hidden1=128,
        hidden2=64,
        output_size=16,
        dropout=0
    ):
        super().__init__()

        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden1,
            batch_first=True
        )

        self.dropout1 = nn.Dropout(dropout)

        self.lstm2 = nn.LSTM(
            input_size=hidden1,
            hidden_size=hidden2,
            batch_first=True
        )

        self.dropout2 = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden2, output_size)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)

        out, _ = self.lstm2(out)
        out = self.dropout2(out)

        out = out[:, -1, :]
        return self.fc(out)

model = cpu_utilizationlstm()    
    

In [13]:
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()


cpu_utilizationlstm(
  (lstm1): LSTM(35, 128, batch_first=True)
  (dropout1): Dropout(p=0, inplace=False)
  (lstm2): LSTM(128, 64, batch_first=True)
  (dropout2): Dropout(p=0, inplace=False)
  (fc): Linear(in_features=64, out_features=16, bias=True)
)

In [14]:
parquet_path = project_root / "Data_analysis" / "ai_multicore_schedular_cleaned.parquet"
df = pd.read_parquet(parquet_path)


In [15]:
cpu_cols = [ f"cpu_{i}_utilization"  for  i in range(16)]
queue_cols = [ f"queue_length_cpu_{i}" for i in  range(16)]
extra_features = ["process_count"  , "running_processes" ,
                    "overall_cpu_utilization" ,
                  ]
feature_cols = cpu_cols + queue_cols + extra_features


In [16]:
sequence_length = 20
if len(df) < sequence_length:
    raise ValueError(f"Need at least {sequence_length} rows for the LSTM input")
x_real = df[feature_cols].tail(sequence_length).to_numpy(dtype=np.float32)
x_scale = feature_scaler.transform(x_real)
x_input = torch.as_tensor(x_scale[np.newaxis, :, :], dtype=torch.float32, device=device)


In [17]:
with torch.no_grad():
    prediction_scaled = model(x_input).cpu().numpy()
prediction = target_scaler.inverse_transform(prediction_scaled)


In [18]:
prediction_utilization = prediction[0].astype(np.float32)
current_utilization = df[cpu_cols].iloc[-1].to_numpy(dtype=np.float32)
queue_length = df[queue_cols].iloc[-1].to_numpy(dtype=np.float32)
def encode_task_features(task_cpu_usage, task_runtime, task_priority, task_wait_time_ms):
    return np.asarray([
        np.clip(float(task_cpu_usage) / 100.0, 0.0, 1.0),
        np.clip(float(task_runtime) / 60.0, 0.0, 1.0),
        np.clip((float(task_priority) + 20.0) / 39.0, 0.0, 1.0),
        np.clip(float(task_wait_time_ms) / 1000.0, 0.0, 1.0),
    ], dtype=np.float32)

task_feature = encode_task_features(0.0, 0.0, 0, 0.0)
state = np.concatenate([prediction_utilization, current_utilization, queue_length, task_feature]).astype(np.float32)


In [19]:
task_cols = [
    "task_cpu_usage",
    "task_runtime",
    "task_priority",
    "task_wait_time_delta_ms",
    "current_core",
    "task_state"
]

In [20]:
class MulticoreSchedulingEnv:
    def __init__(self, num_core=NUM_CORES):
        if num_core != NUM_CORES:
            raise ValueError(f"This network is configured for {NUM_CORES} cores")
        self.num_cores = num_core
        self.reset()

    def reset(self):
        self.queue_lengths = np.zeros(self.num_cores, dtype=np.float32)
        self.current_utilization = np.zeros(self.num_cores, dtype=np.float32)
        self.predicted_utilization = np.zeros(self.num_cores, dtype=np.float32)
        self.task_features = np.zeros(TASK_FEATURES, dtype=np.float32)
        return self.__get_state__()

    def __get_state__(self):
        return np.concatenate([self.predicted_utilization, self.current_utilization,
                               self.queue_lengths, self.task_features]).astype(np.float32)

    def set_observation(self, predicted_utilization, current_utilization, queue_lengths, task_features):
        values = {
            "predicted_utilization": np.asarray(predicted_utilization, dtype=np.float32),
            "current_utilization": np.asarray(current_utilization, dtype=np.float32),
            "queue_lengths": np.asarray(queue_lengths, dtype=np.float32),
            "task_features": np.asarray(task_features, dtype=np.float32),
        }
        expected = {"predicted_utilization": (NUM_CORES,), "current_utilization": (NUM_CORES,),
                    "queue_lengths": (NUM_CORES,), "task_features": (TASK_FEATURES,)}
        for name, value in values.items():
            if value.shape != expected[name] or not np.isfinite(value).all():
                raise ValueError(f"{name} must have shape {expected[name]} and finite values")
        self.predicted_utilization = values["predicted_utilization"]
        self.current_utilization = values["current_utilization"]
        self.queue_lengths = values["queue_lengths"]
        self.task_features = values["task_features"]
        return self.__get_state__()

env = MulticoreSchedulingEnv()


In [21]:
def calculate_reward(predicted_utilization, current_utilization, queue_lengths, selected_core):
    predicted = np.clip(np.asarray(predicted_utilization, dtype=np.float32), 0, 100) / 100.0
    current = np.clip(np.asarray(current_utilization, dtype=np.float32), 0, 100) / 100.0
    queues = np.maximum(np.asarray(queue_lengths, dtype=np.float32), 0)
    normalized_queues = queues / max(float(queues.max()), 1.0)
    estimated_load = 0.5 * predicted + 0.3 * current + 0.2 * normalized_queues
    return -float(estimated_load[selected_core])


In [22]:
def assign_task(env, selected_core):
    if not 0 <= selected_core < env.num_cores:
        raise ValueError("Invalid CPU core index")
    env.queue_lengths[selected_core] += 1.0
    reward = calculate_reward(env.predicted_utilization, env.current_utilization,
                              env.queue_lengths, selected_core)
    return env.__get_state__().copy(), reward


In [23]:
from  dataclasses  import dataclass
@dataclass 
class SimTask : 
    task_id : int
    remaining_runtime : float 
    priority : int  =  0 
    waiting_time : float = 0.0 
    assigned_core  : int  = -1 

In [24]:
core_queues = [deque() for  _ in range(16) ]
def enqueue_task ( task , selected_core) : 
    if not 0 <= selected_core < 16 : 
        raise ValueError("Core must be between 0 to 15")
    task.assigned_core = selected_core
    core_queues[selected_core].append(task)
    return len(core_queues[selected_core])

In [25]:
def simulate_time_step( core_queues , time_step = 1.0) : 
    if  time_step <= 0 : 
        raise ValueError("time_step must be positive")
    
    completed_tasks = []
    executed_tasks   = []
    
    for core_id , queue in enumerate(core_queues) : 
        
        if not queue : 
            continue
        
        running_task = queue[0]
        running_task.remaining_runtime  -= time_step
        executed_tasks.append(running_task.task_id)
        
        for waiting_task  in  list(queue) [ 1 : ] : 
            waiting_task.waiting_time += time_step
        if running_task.remaining_runtime <= 0 : 
            finished = queue.popleft()
            completed_tasks.append(finished.task_id)
    return {
        "executed_tasks" : executed_tasks , 
        "completed_tasks" : completed_tasks , 
        "queue_lengths" : [len(q) for q in core_queues]
    }        

result = simulate_time_step(core_queues, time_step=1.0)          

In [26]:
def get_queue_lengths(core_queues) : 
    return np.array(
        [len(queue) for queue  in core_queues] , 
        dtype= np.float32
    )
env.queue_lengths =  get_queue_lengths(core_queues)

assert np.array_equal(
    env.queue_lengths, 
    get_queue_lengths(core_queues)
)

In [27]:
def assign_sim_task(task, selected_core, core_queues):
    if not 0 <= selected_core < len(core_queues):
        raise ValueError("Invalid CPU core index")
    if any(existing.task_id == task.task_id for queue in core_queues for existing in queue):
        raise ValueError(f"Task {task.task_id} is already in a queue")
    task.assigned_core = selected_core
    core_queues[selected_core].append(task)
    return len(core_queues[selected_core])


In [28]:
random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
core_queues = [deque() for _ in range(NUM_CORES)]
replay_buffer = ReplayBuffer(capacity=10000)
env.set_observation(prediction_utilization, current_utilization,
                    np.zeros(NUM_CORES, dtype=np.float32),
                    encode_task_features(0, 0, 0, 0))
target_size = 2000
task_id = 0

for _ in range(target_size):
    task_cpu = float(np.random.uniform(1.0, 100.0))
    task_runtime = float(np.random.uniform(2.0, 60.0))
    task_priority = int(np.random.randint(-20, 20))
    task_wait_ms = float(np.random.uniform(0.0, 1000.0))
    
    task_features = encode_task_features(task_cpu, task_runtime, task_priority, task_wait_ms)
    env.queue_lengths = get_queue_lengths(core_queues)
    state_before = env.set_observation(prediction_utilization, current_utilization,
                                       env.queue_lengths, task_features).copy()

    epsilon = max(0.05, 1.0 - 0.95 * (task_id / target_size))
    selected_core = select_action(policy_net, state_before, epsilon=epsilon)
    task = SimTask(task_id=task_id, remaining_runtime=task_runtime, priority=task_priority)
    assign_sim_task(task, selected_core, core_queues)

    queue_after_assignment = get_queue_lengths(core_queues)
    reward = calculate_reward(prediction_utilization, current_utilization,
                              queue_after_assignment, selected_core)
    simulate_time_step(core_queues, time_step=1.0)

    next_task_features = encode_task_features(
        np.random.uniform(1.0, 100.0), np.random.uniform(2.0, 60.0),
        np.random.randint(-20, 20), np.random.uniform(0.0, 1000.0))
    next_state = env.set_observation(prediction_utilization, current_utilization,
                                     get_queue_lengths(core_queues), next_task_features).copy()
    replay_buffer.add(state_before, selected_core, reward, next_state, False,
                      next_valid_actions=range(NUM_CORES))
    task_id += 1


In [29]:
batch_size = 64
loss = train_step(policy_net, target_net, optimizer, replay_buffer, batch_size=batch_size, gamma=gamma)
loss


0.7505136132240295

In [30]:
num_training_steps = 1000
target_update_frequency = 100
loss_history = []
for step in range(1, num_training_steps + 1):
    loss = train_step(policy_net, target_net, optimizer, replay_buffer,
                      batch_size=64, gamma=gamma)
    if loss is None:
        break
    loss_history.append(loss)
    if step % target_update_frequency == 0:
        target_net.load_state_dict(policy_net.state_dict())
    if step % 100 == 0:
        print(f"Step {step}/{num_training_steps} | Loss: {loss:.6f}")


Step 100/1000 | Loss: 0.655843
Step 200/1000 | Loss: 0.139369
Step 300/1000 | Loss: 0.082230
Step 400/1000 | Loss: 0.056486
Step 500/1000 | Loss: 0.017422
Step 600/1000 | Loss: 0.000714
Step 700/1000 | Loss: 0.000627
Step 800/1000 | Loss: 0.000616
Step 900/1000 | Loss: 0.000295
Step 1000/1000 | Loss: 0.000410


In [31]:

torch.save(policy_net.state_dict(), "dqn_schedular_fnf.pth")

In [32]:
live_task_features = encode_task_features(task_cpu_usage=25.0, task_runtime=5.0,
                                           task_priority=0, task_wait_time_ms=10.0)
live_state = env.set_observation(prediction_utilization, current_utilization,
                                 get_queue_lengths(core_queues), live_task_features)
policy_net.eval()
selected_core = select_action(policy_net, live_state, epsilon=0.0,
                              valid_actions=range(NUM_CORES))
print("Selected core:", selected_core)

Selected core: 13
